---
layout: post
toc: true
title: Scope and Access
menu: nav/csa_units/csaunit3.html
permalink: /csa/unit_03/3_8
---

# Scope and Access

Two closely related questions come up constantly once a program has more than one class: where is this variable visible, and who is allowed to touch it? This lesson covers variable scope, access modifiers, and the `this` keyword you've already been using in every constructor.

<div class="callout callout-objectives">
<h4>Objectives</h4>

- Distinguish instance variable scope, local variable scope, and parameter scope
- Choose the right access modifier (`public`, `private`, `protected`, default) for a field or method
- Use `this` to resolve a shadowed field name and to call one constructor from another
- Recognize the classic bug caused by forgetting `this` when a parameter shadows a field

</div>

<style>
.lesson-wrap { font-family: 'Segoe UI', system-ui, -apple-system, sans-serif; color:#1e293b; line-height:1.65; }
.lesson-wrap h2 { color:#0f172a; border-bottom:2px solid #e2e8f0; padding-bottom:6px; margin-top:1.8em; }
.lesson-wrap h3 { color:#1e293b; margin-top:1.4em; }
.callout { border-radius:10px; padding:16px 20px; margin:16px 0; border:1px solid #e2e8f0; background:#f8fafc; }
.callout-objectives { background:#eff6ff; border-left:4px solid #2563eb; }
.callout-objectives h4 { color:#1d4ed8; margin:0 0 8px 0; }
.callout-hint { background:#fefce8; border-left:4px solid #ca8a04; }
.callout-hint h4 { color:#a16207; margin:0 0 8px 0; }
.callout-practice { background:#f0fdf4; border-left:4px solid #16a34a; }
.callout-practice h4 { color:#15803d; margin:0 0 8px 0; }
.callout-homework { background:#fdf4ff; border-left:4px solid #9333ea; }
.callout-homework h4 { color:#7e22ce; margin:0 0 8px 0; }
.callout-note { background:#f1f5f9; border-left:4px solid #64748b; }
.callout-note h4 { color:#334155; margin:0 0 8px 0; }
table { border-collapse:collapse; width:100%; margin:16px 0; }
th, td { border:1px solid #e2e8f0; padding:10px 14px; text-align:left; }
th { background:#f1f5f9; color:#0f172a; }
details { border:1px solid #e2e8f0; border-radius:8px; padding:10px 16px; margin:12px 0; background:#fafafa; }
details summary { cursor:pointer; font-weight:600; color:#2563eb; }
details[open] summary { margin-bottom:10px; }
.btn { display:inline-block; padding:10px 22px; border-radius:8px; border:none; font-weight:600; cursor:pointer; font-size:14px; transition:background .2s,transform .2s; }
.btn-primary { background:#2563eb; color:#fff; }
.btn-primary:hover { background:#1d4ed8; }
.btn-secondary { background:#e2e8f0; color:#1e293b; }
.btn-secondary:hover { background:#cbd5e1; }
.pill { display:inline-block; padding:3px 12px; border-radius:999px; background:#eff6ff; color:#1d4ed8; font-size:0.85em; font-weight:600; margin-right:6px; }
</style>

## Variable Scope

Scope is the region of code where a variable's name is valid.

- **Instance variables** are declared inside a class but outside any method. They're visible to every method in the class and last as long as the object does.
- **Local variables** are declared inside a method or block. They only exist inside that method or block, and disappear once it finishes running.
- **Parameters** work like local variables scoped to the method they belong to.

In [ ]:
public class Shape {
    private int length = 10; // instance variable: visible to every method below

    public void grow() {
        int extra = 5; // local variable: only exists inside grow()
        length = length + extra;
    }

    public void printLength() {
        System.out.println(length); // fine, length is an instance variable
        // System.out.println(extra); // would NOT compile, extra is out of scope here
    }
}

Shape s = new Shape();
s.grow();
s.printLength(); // 15

## Access Modifiers

Scope controls where a name is *visible*. Access modifiers control who is *allowed to use it* once it's visible.

| Modifier | Visible from |
|---|---|
| `public` | Any class, anywhere |
| `protected` | The same package, plus subclasses in other packages |
| (no modifier / default) | The same package only |
| `private` | Only inside the same class |

This is exactly the tool behind encapsulation from Impact of Program Design: mark fields `private` so nothing outside the class can set them directly, and expose only the specific `public` methods that are allowed to change them, with whatever validation those methods enforce. Most fields should default to `private`. Most methods that make up a class's usable interface should be `public`.

## The `this` Keyword

`this` is a reference to the current object, the specific object whose method or constructor is currently running. You've already used it in every constructor so far without a dedicated explanation, so here is why it was there.

### Fixing Shadowing

A parameter or local variable can have the same name as an instance field. When that happens, the field is "shadowed," meaning the plain name refers to the parameter, not the field, anywhere inside that block. `this.fieldName` reaches past the shadow to the field.

<div class="callout callout-hint">
<h4>Common Pitfall</h4>

Forgetting `this` when a parameter shadows a field doesn't cause a compiler error, which is exactly what makes it dangerous. It just silently does nothing.

</div>

In [ ]:
public class Shape {
    private String name;

    // BUGGY: parameter "name" shadows the field "name"
    public Shape(String name) {
        name = name; // this just assigns the parameter to itself, the field is never touched
    }

    public String getName() {
        return this.name; // still whatever the field's default was: null
    }
}

Shape s = new Shape("triangle");
System.out.println(s.getName()); // null, not "triangle"

In [ ]:
public class Shape {
    private String name;

    // FIXED: this.name is the field, name is the parameter
    public Shape(String name) {
        this.name = name;
    }

    public String getName() {
        return this.name;
    }
}

Shape s = new Shape("triangle");
System.out.println(s.getName()); // "triangle"

### Calling Another Constructor with `this(...)`

`this` has one more job: calling a different constructor in the same class, so you don't have to repeat setup logic. `this(...)` must be the very first line of the constructor.

In [ ]:
public class Shape {
    protected String name;
    private int length;
    private int width;

    public Shape() {
        this("shape", 10, 5); // calls the constructor below instead of repeating the assignments
    }

    public Shape(String name, int length, int width) {
        this.name = name;
        this.length = length;
        this.width = width;
    }

    public void print_shape() {
        System.out.println(name + ": " + length + "x" + width);
    }
}

Shape s1 = new Shape();
s1.print_shape(); // shape: 10x5

`this` also shows up as a return value: the chained `Counter` example from the last lesson returned `this` from `increment()` so calls could be strung together. In every case, `this` means the same thing: the object currently running the code.

<div class="callout callout-practice">
<h4>Popcorn Hack</h4>

Fix the constructor below so `getWidth()` returns 8 instead of 0.

```java
public class Shape {
    private int width;

    public Shape(int width) {
        width = width; // bug is here
    }

    public int getWidth() {
        return width;
    }
}

Shape s = new Shape(8);
System.out.println(s.getWidth());
```

<details>
<summary>Show answer</summary>

Change `width = width;` to `this.width = width;`. The parameter `width` shadows the field `width`, so without `this`, the assignment just reassigns the parameter to itself and the field is never set.

</details>

</div>

We've now covered every piece needed to write a single class well. The last topic in this unit, polymorphism, is about what happens when several classes related by inheritance are in play at once, and a single line of code can behave differently depending on which object is actually running it.